In [ ]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import scipy.io.wavfile
import torch
from fastai.vision import *
from fastai.metrics import error_rate

import IPython.display as ipd
from tqdm import tqdm_notebook

In [ ]:
import re
from fastai.vision.all import *
# Regular expression to capture the gender label
pat = re.compile(r'.*/(\w+)_(\w+)_(\d+_\d+-\d+-\d+-\d+-\d+-\d+-\d+)\.jpg')



In [ ]:
from pathlib import Path
import numpy as np

path_img = "mfc_data"
fnames = list(Path(path_img).rglob('*.jpg')) + list(Path(path_img).rglob('*.jpeg')) + list(Path(path_img).rglob('*.png'))
np.random.seed(42)
np.random.shuffle(fnames)
fnames[:5]


In [ ]:
female_count = 0
male_count = 0

for fname in fnames:
    label = fname.stem.split('_')[0]
    if label == 'Female':
        female_count += 1
    elif label == 'Male':
        male_count += 1

print(f"Female count: {female_count}")
print(f"Male count: {male_count}")

In [ ]:
# Tạo DataBlock
dblock = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_items=get_image_files,
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    get_y=RegexLabeller(pat, match=1)
)

# Tạo DataLoaders
dls = dblock.dataloaders(path_img, bs=64)

# Hiển thị batch và classes
dls.show_batch()
print(dls.vocab)



In [ ]:
# Replace the old show_batch code with the following
dls.show_batch(nrows=3, ncols=3, figsize=(7,6))


In [ ]:
print(dls.vocab)
print(len(dls.vocab), dls.c)


In [ ]:
learn = cnn_learner(dls, models.resnet34, metrics=error_rate)

In [ ]:
learn.lr_find()

In [ ]:
learn.recorder.plot_lr_find()

In [ ]:

# Huấn luyện mô hình
learn.fit_one_cycle(2, 2e-3)


In [ ]:

# Xuất mô hình sau bước 1
learn.export('vgg16_step1.pkl')


In [ ]:
learn.unfreeze()

In [ ]:
learn.fit_one_cycle(1)

In [ ]:

# Xuất mô hình sau bước 2
learn.export('vgg16_step2.pkl')